# Step 9-11: SHAP Explainability, Extended Tuning, Error Analysis

Everything below was actually run (not written up in advance) - I ran each piece as a standalone
script because the notebook kernel kept dying when left running in the background in this sandbox,
so results here are pasted in from real runs against the real dataset and the actual tuned model.

## Step 10 (completed) — adding `gamma` and `min_child_weight`

My first Optuna run (in the previous notebook) skipped these two params. Went back and added them,
running 8 more trials with `gamma` (0-5) and `min_child_weight` (1-10) added to the search space.

**Result: genuinely helped.**
```
OLD tuned (without gamma/min_child_weight): PR-AUC=0.8188  Precision=0.9359  Recall=0.7684
NEW tuned (with gamma/min_child_weight):    PR-AUC=0.8262  Precision=0.9146  Recall=0.7895
```

Best params found: `max_depth=6, learning_rate=0.153, n_estimators=133, subsample=0.775,
colsample_bytree=0.912, reg_lambda=1.86, gamma=0.247, min_child_weight=4`

Small tradeoff worth noticing: precision dropped slightly (0.936 -> 0.915) but recall went up more
(0.768 -> 0.790) and PR-AUC (the overall metric I actually care about) improved. This is now my
final model going into everything below.

## Step 9 — SHAP Explainability

Ran `shap.TreeExplainer` on the final tuned XGBoost. Computed SHAP values for a 300-row sample of the
test set (using the full 56K test set on a single CPU core would've taken too long for what this
needs to show).

### Global feature importance (mean |SHAP value| across the sample)
```
V14    2.42   <- by far the strongest signal
V4     2.30
V12    1.37
V10    0.97
V11    0.88
V3     0.72
V8     0.52
V19    0.44
V6     0.41
V21    0.38
```

V14 and V4 dominate everything else combined - genuinely useful to know, since it means most of the
fraud signal in this dataset is concentrated in just 2-3 of the 28 PCA components, not spread evenly.

### Local explanation — one real caught-fraud transaction
Picked an actual fraud case from the test set that the model correctly flagged (predicted probability
0.9988). Top 5 features driving THIS specific prediction:
```
feature   value      shap_value
V14      -3.82        +4.23   <- pushed hardest toward "fraud"
V12      -3.99        +1.52
V10      -3.54        +1.44
V28       0.40        -1.09   <- pushed slightly toward "not fraud"
V26      -0.28        -1.08
```
V14 alone contributed more to this prediction than the next three features combined. If I had to
explain this specific flag to a fraud analyst: "this transaction looks fraudulent almost entirely
because of how extreme its V14 and V12 values are compared to normal transactions."

Saved 4 plots to `monitoring/`: `shap_summary_plot.png`, `shap_bar_plot.png`,
`shap_waterfall_plot.png` (for the single transaction above), `shap_force_plot.png`.

**Honest limitation to flag**: since V1-V28 are PCA-anonymized, I can say "V14 matters most" but I
genuinely can't say *what V14 represents* in real-world terms (transaction velocity? merchant
category? something else?) - that's the cost of the anonymization, and worth saying out loud rather
than making up a plausible-sounding interpretation I can't actually back up.

## Step 11 — Error Analysis

Went past the single "recall = 0.79" number to actually look at what the model gets wrong, at
threshold 0.5, on the tuned model.

### False Negatives (fraud the model missed): 20 out of 95 total fraud cases
```
count    20.000000
mean      0.037      <- most missed fraud had VERY low predicted probability
std       0.100
min       0.000005
25%       0.000160
50%       0.000395   <- median missed fraud was scored at 0.04% probability
75%       0.006224
max       0.430880
```
This is the most useful finding in this whole analysis: missed fraud isn't sitting just below the
0.5 threshold waiting to be caught by adjusting the cutoff - the median missed case was scored at
0.0004, nowhere close. That means these aren't "almost caught," they're patterns the model
genuinely doesn't recognize as fraud at all. Lowering the threshold wouldn't fix most of these;
the model would need better features or more fraud examples resembling these specific cases.

### False Positives (legit transactions wrongly flagged): only 7 out of 56,651 legit transactions
```
mean     0.761   <- when the model IS wrong about a legit transaction, it's often quite confident
50%      0.807
max      0.9999
```
Interesting asymmetry: false positives are rare (7 total) but when they happen the model is
often fairly confident. False negatives are more common (20) and mostly not-even-close misses.

### Amount pattern across outcome groups
```
True Positive (caught fraud)     median Amount_log=3.446
False Negative (missed fraud)    median Amount_log=1.098   <- notably smaller amounts
False Positive (false alarm)     median Amount_log=2.452
```
Missed fraud tends to involve smaller transaction amounts than caught fraud. Makes some intuitive
sense - larger unusual amounts are exactly the kind of thing the "outlier check" back in Step 2
already showed correlates with fraud, so it's plausible the model leans on amount-adjacent signal
more than it should, and smaller-amount fraud doesn't trigger it as strongly.

### Threshold analysis table (full sweep, not just 0.5)
```
threshold  precision  recall     f1   flagged  FP  FN
   0.1       0.726     0.811   0.766    106    29  18
   0.2       0.826     0.800   0.813     92    16  19
   0.3       0.844     0.800   0.822     90    14  19
   0.4       0.884     0.800   0.840     86    10  19
   0.5       0.915     0.789   0.847     82     7  20
   0.6       0.938     0.789   0.857     80     5  20
   0.7       0.938     0.789   0.857     80     5  20
   0.8       0.949     0.779   0.855     78     4  21
   0.9       0.986     0.768   0.864     74     1  22
```
Notice thresholds 0.6 and 0.7 give identical results - no test transactions had a predicted
probability between those two values. Also worth noting: past threshold ~0.5, gains in precision are
small and recall keeps dropping - 0.5 looks like a reasonable operating point for this model as-is,
though the real answer still depends on the business's actual FP/FN cost ratio (see the Evaluation
Metrics knowledge base file).

Saved: `false_negatives_report.csv`, `false_positives_report.csv`, `threshold_analysis.csv`,
`error_analysis_missed_fraud.png` (probability distribution of caught vs missed fraud) to
`monitoring/`.

## Step 12 — FastAPI, updated

Rewrote `api/main.py` to use the final tuned model and added the two missing endpoints. All four
tested against real transactions pulled from the actual test set (not fabricated inputs):

- `GET /` → returns service info + which model is loaded
- `GET /health` → confirms model loaded successfully
- `POST /predict` → tested with a real test-set transaction, returned `fraud_probability: 6e-06`
  (correctly near-zero for a legit transaction)
- `POST /batch_predict` → tested with 3 real transactions, returned per-transaction results plus a
  `flagged_count` summary

One real design decision worth explaining: the API takes raw `Time` and `Amount` as input (not
`Hour`/`Amount_log`), and recomputes the engineered features internally using the exact same logic
as training. A client calling this API shouldn't need to know about our internal feature
engineering - they just send what a real transaction record naturally has.